# PrimeNet Fig.5 — mimic_all finetune (Colab, no Drive)

Finetune **mimicall** scenarios using the HPC `mimic_all` SSL checkpoint. **No Google Drive.**

| Scenario | Meaning |
|----------|---------|
| `mimicall_all` | pretrain on mimic_all, finetune all layers |
| `mimicall_final` | pretrain on mimic_all, freeze backbone |

**Runtime:** GPU required. Full run ≈ hours (2 scenarios × 5 folds).

**Branch:** `primenet`

## What goes where

Repo root on Colab: `/content/ChemoTreeVsDL/`

### A) HPC checkpoint (required)

From FAU box `primenet_checkpoint`:

- `checkpoint_best.bin`
- `primenet_saved_variables.pkl`

Must end up at:

```text
MIMIC_IV/saved_data/results/mimic_all/time_series/pretrain/primenet/fig5_pt_mimicall/fold_0/grid_none/
├── checkpoint_best.bin
└── primenet_saved_variables.pkl
```

### B) NF data (required)

Either:

1. A zip/tarball of `MIMIC_IV/saved_data/` (from a previous Colab/HPC run), **or**
2. The two raw NF CSVs → run prepare (cell 4b)

Minimum prepared files:

```text
MIMIC_IV/saved_data/
├── cohorts/mimic_cohort_NF_30_days.csv.gz
├── folds/mimic_cohort_NF_30_days/fold_{0..4}.pkl
├── top_features/mimic_top100_features.pkl
└── processed_admission_features_for_ts/mimic_cohort_NF_30_days/
    └── ..._admissions_labs_14_days_to_ts.csv.gz
```

### Transfer options (pick one)

| Option | Best for | How |
|--------|----------|-----|
| **1. Direct upload** | Checkpoint (~few hundred MB) + small zips | Colab left sidebar → Files, or `files.upload()` |
| **2. Download URL** | FAU box / Dropbox / Zenodo / any HTTPS link | Paste share URL in options cell |
| **3. Raw CSVs + prepare** | If you only have the two NF CSVs | Upload CSVs, run prepare |

You do **not** need full `mimic_all` labs on Colab.

## 1. Clone repo + install

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "AhmedSofan10/ChemoTreeVsDL"
BRANCH = "primenet"
REPO_DIR = "/content/ChemoTreeVsDL"

if not Path(REPO_DIR).is_dir():
    subprocess.check_call(
        ["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR]
    )
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
root = Path.cwd()
sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

import torch
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ready:", root)

## 2. Options — choose how data arrives

In [ ]:
from pathlib import Path

RUN_FAST = False  # True = smoke timings; False = full settings
MIMIC_ALL_COHORT = "mimic_all"

# How to get files into Colab:
#   "upload"  — use cells 3a / 3b (browser upload)
#   "url"     — download from HTTPS links below (FAU box share, Dropbox, Zenodo, …)
#   "prepare" — only raw NF CSVs; build saved_data with --phase prepare
DATA_MODE = "upload"  # "upload" | "url" | "prepare"

# --- used only if DATA_MODE == "url" ---
# Direct download links (must be publicly reachable from Colab).
# Tip: FAU box / Dropbox often need a "direct download" / shared-link URL, not the web viewer page.
CKPT_ZIP_URL = ""          # zip/tarball with checkpoint_best.bin + primenet_saved_variables.pkl
SAVED_DATA_ZIP_URL = ""    # zip/tarball of MIMIC_IV/saved_data (or its contents)
# Or set individual file URLs instead of CKPT_ZIP_URL:
CKPT_BIN_URL = ""
CKPT_PKL_URL = ""

# Staging dirs for uploads (Colab Files sidebar → /content/...)
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_MODE =", DATA_MODE)
print("Upload staging:", UPLOAD_DIR)

## 3a. Checkpoint — direct upload (if `DATA_MODE == "upload"`)

**Easiest:** zip the two files on your laptop as `primenet_checkpoint.zip`, then either:

- Left sidebar **Files** → upload to `/content/uploads/`, **or**
- Run this cell and pick the zip (or the two files) in the dialog.

Expected names: `checkpoint_best.bin`, `primenet_saved_variables.pkl` (loose or inside a zip).

In [ ]:
import shutil
import zipfile
from pathlib import Path

if DATA_MODE != "upload":
    print("Skipped (DATA_MODE != upload)")
else:
    from google.colab import files

    print("Upload checkpoint zip OR the two files (checkpoint_best.bin, primenet_saved_variables.pkl)")
    uploaded = files.upload()  # saves into cwd = repo root
    for name in uploaded:
        src = Path(name)
        dst = UPLOAD_DIR / src.name
        if src.resolve() != dst.resolve():
            shutil.move(str(src), dst)
        print("staged", dst)

    # unpack any zips in upload dir
    for zpath in UPLOAD_DIR.glob("*.zip"):
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(UPLOAD_DIR / zpath.stem)
        print("extracted", zpath.name)

    print("Upload dir contents:")
    for p in sorted(UPLOAD_DIR.rglob("*")):
        if p.is_file():
            print(" ", p.relative_to(UPLOAD_DIR), f"({p.stat().st_size/1e6:.1f} MB)")

## 3b. NF `saved_data` — direct upload (if `DATA_MODE == "upload"`)

Upload a zip of your previous `MIMIC_IV/saved_data` (e.g. `saved_data.zip`).

If the zip is large and the dialog is slow, use the **Files** sidebar and upload to `/content/uploads/saved_data.zip`, then run the next *place* cell only.

In [ ]:
import shutil
import zipfile
from pathlib import Path

if DATA_MODE != "upload":
    print("Skipped (DATA_MODE != upload)")
else:
    # Prefer sidebar upload to /content/uploads/saved_data.zip to avoid huge browser dialogs.
    already = list(UPLOAD_DIR.glob("*saved_data*.zip")) + list(UPLOAD_DIR.glob("saved_data.zip"))
    if already:
        print("Found existing zip(s):", [p.name for p in already])
    else:
        from google.colab import files

        print("Upload saved_data.zip (or skip this cell if you already put it in /content/uploads/)")
        uploaded = files.upload()
        for name in uploaded:
            src = Path(name)
            dst = UPLOAD_DIR / src.name
            if src.resolve() != dst.resolve():
                shutil.move(str(src), dst)
            print("staged", dst)

    for zpath in UPLOAD_DIR.glob("*.zip"):
        out = UPLOAD_DIR / zpath.stem
        if out.exists() and any(out.rglob("*.csv.gz")):
            continue
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(out)
        print("extracted", zpath.name, "→", out)

## 3c. Download from URL (if `DATA_MODE == "url"`)

Set `CKPT_ZIP_URL` / `SAVED_DATA_ZIP_URL` (or per-file URLs) in the options cell, then run this.

In [ ]:
import subprocess
import zipfile
from pathlib import Path
from urllib.parse import urlparse

def _download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    print("GET", url, "→", dest)
    subprocess.check_call(["wget", "-q", "--show-progress", "-O", str(dest), url])

if DATA_MODE != "url":
    print("Skipped (DATA_MODE != url)")
else:
    if CKPT_ZIP_URL:
        z = UPLOAD_DIR / "primenet_checkpoint.zip"
        _download(CKPT_ZIP_URL, z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(UPLOAD_DIR / "primenet_checkpoint")
    else:
        if not (CKPT_BIN_URL and CKPT_PKL_URL):
            raise ValueError("Set CKPT_ZIP_URL or both CKPT_BIN_URL and CKPT_PKL_URL")
        _download(CKPT_BIN_URL, UPLOAD_DIR / "checkpoint_best.bin")
        _download(CKPT_PKL_URL, UPLOAD_DIR / "primenet_saved_variables.pkl")

    if SAVED_DATA_ZIP_URL:
        z = UPLOAD_DIR / "saved_data.zip"
        _download(SAVED_DATA_ZIP_URL, z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(UPLOAD_DIR / "saved_data")
    else:
        print("No SAVED_DATA_ZIP_URL — use DATA_MODE='prepare' or upload saved_data separately")

    print("Downloads ready under", UPLOAD_DIR)

## 3d. Raw NF CSVs only (if `DATA_MODE == "prepare"`)

Upload:

- `mimic_cohort_NF_30_days.csv`
- `mimic_cohort_NF_30_days_admissions_labs_14_days.csv`

into `data/raw/`. Checkpoint still required (use 3a or 3c).

In [ ]:
from pathlib import Path

if DATA_MODE != "prepare":
    print("Skipped (DATA_MODE != prepare)")
else:
    from google.colab import files

    raw = Path("data/raw")
    raw.mkdir(parents=True, exist_ok=True)
    print("Upload the two NF CSV files")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = raw / name
        dest.write_bytes(data)
        print("Saved", dest, f"({dest.stat().st_size // 1024} KB)")

    required = [
        "mimic_cohort_NF_30_days.csv",
        "mimic_cohort_NF_30_days_admissions_labs_14_days.csv",
    ]
    missing = [f for f in required if not (raw / f).is_file()]
    if missing:
        raise FileNotFoundError(f"Still missing in data/raw/: {missing}")

## 4. Place files into the exact repo paths

In [ ]:
import shutil
import zipfile
from pathlib import Path

root = Path.cwd()
assert root.name == "ChemoTreeVsDL", root

# Unzip any *.zip under /content/uploads (e.g. sidebar upload of saved_data.zip)
for zpath in sorted(UPLOAD_DIR.glob("*.zip")):
    out = UPLOAD_DIR / zpath.stem
    print(f"Extracting {zpath.name} → {out}")
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(out)

ckpt_dst = (
    root
    / "MIMIC_IV"
    / "saved_data"
    / "results"
    / MIMIC_ALL_COHORT
    / "time_series"
    / "pretrain"
    / "primenet"
    / "fig5_pt_mimicall"
    / "fold_0"
    / "grid_none"
)
ckpt_dst.mkdir(parents=True, exist_ok=True)


def _find_one(name: str) -> Path:
    hits = [p for p in UPLOAD_DIR.rglob(name) if p.is_file()]
    if not hits:
        raise FileNotFoundError(
            f"Could not find {name} under {UPLOAD_DIR}. "
            "Upload/download the checkpoint first (cells 3a or 3c)."
        )
    return hits[0]


# --- checkpoint ---
for name in ("checkpoint_best.bin", "primenet_saved_variables.pkl"):
    src = _find_one(name)
    shutil.copy2(src, ckpt_dst / name)
    print(f"checkpoint: {src} → {ckpt_dst / name}")

log_hits = list(UPLOAD_DIR.rglob("log.txt"))
if log_hits:
    shutil.copy2(log_hits[0], ckpt_dst / "log.txt")

# --- saved_data (skip if prepare mode) ---
dst_saved = root / "MIMIC_IV" / "saved_data"
if DATA_MODE == "prepare":
    print("DATA_MODE=prepare → will build saved_data in next cell")
else:
    candidates = []
    for p in UPLOAD_DIR.rglob("cohorts"):
        if p.is_dir() and (p.parent / "folds").is_dir():
            candidates.append(p.parent)
    if not candidates:
        raise FileNotFoundError(
            f"No extracted saved_data under {UPLOAD_DIR}. "
            "Need saved_data.zip unzipped (or set DATA_MODE='prepare')."
        )
    src_saved = candidates[0]
    dst_saved.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_saved, dst_saved, dirs_exist_ok=True)
    print("saved_data:", src_saved, "→", dst_saved)

print("\nCheckpoint dir:", ckpt_dst)

## 4b. Prepare NF saved_data from raw CSVs (only if `DATA_MODE == "prepare"`)

In [ ]:
import subprocess
import sys

if DATA_MODE != "prepare":
    print("Skipped")
else:
    cmd = [sys.executable, "colab_primenet_fig5.py", "--phase", "prepare"]
    print("+", " ".join(cmd), flush=True)
    rc = subprocess.call(cmd)
    if rc != 0:
        raise RuntimeError(f"prepare failed: {rc}")

## 5. Preflight checks

In [ ]:
from pathlib import Path

NF = "mimic_cohort_NF_30_days"
saved = Path("MIMIC_IV/saved_data")
ckpt = (
    saved
    / "results"
    / MIMIC_ALL_COHORT
    / "time_series"
    / "pretrain"
    / "primenet"
    / "fig5_pt_mimicall"
    / "fold_0"
    / "grid_none"
)

checks = [
    saved / "cohorts" / f"{NF}.csv.gz",
    saved / "top_features" / "mimic_top100_features.pkl",
    saved
    / "processed_admission_features_for_ts"
    / NF
    / f"{NF}_admissions_labs_14_days_to_ts.csv.gz",
    *[saved / "folds" / NF / f"fold_{i}.pkl" for i in range(5)],
    ckpt / "checkpoint_best.bin",
    ckpt / "primenet_saved_variables.pkl",
]

missing = [str(p) for p in checks if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing:\n  - " + "\n  - ".join(missing))

print("OK — NF data + mimic_all checkpoint present")
print("ckpt:", ckpt.resolve())
for p in (ckpt / "checkpoint_best.bin", ckpt / "primenet_saved_variables.pkl"):
    print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

## 6. Finetune mimicall (2 scenarios × 5 folds)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "colab_primenet_fig5.py",
    "--phase", "finetune",
    "--scenarios", "mimicall",
    "--mimic-all-cohort", MIMIC_ALL_COHORT,
    "--skip-prepare",
    "--skip-extract-mimic-all",
]
if RUN_FAST:
    cmd.append("--fast")

print("+", " ".join(cmd), flush=True)
rc = subprocess.call(cmd)
if rc != 0:
    raise RuntimeError(f"finetune failed with exit {rc}")
print("Finetune finished.")

## 7. Collect metrics + download results zip (no Drive)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import files

subprocess.check_call([sys.executable, "colab_primenet_fig5.py", "--phase", "collect"])

results_root = Path("MIMIC_IV/saved_data/results/mimic_cohort_NF_30_days/time_series")
out_zip = Path("/content/mimicall_finetune_results")

# Zip only mimicall result trees + any collected summary if present
to_zip = []
for name in ("fig5_mimicall_all", "fig5_mimicall_final"):
    # results live under finetune/primenet/<prefix>/
    hits = list(results_root.rglob(name))
    to_zip.extend(hits)

summary = Path("MIMIC_IV/saved_data/results")
for p in summary.rglob("*fig5*summary*"):
    to_zip.append(p)
for p in summary.rglob("*collect*"):
    to_zip.append(p)

staging = Path("/content/results_staging")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

if to_zip:
    for src in to_zip:
        rel = src.relative_to(Path("MIMIC_IV/saved_data")) if "saved_data" in str(src) else src.name
        dest = staging / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dest)
else:
    # fallback: whole results tree for NF cohort
    shutil.copytree(results_root, staging / "time_series", dirs_exist_ok=True)

archive = shutil.make_archive(str(out_zip), "zip", root_dir=staging)
print("Created", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
files.download(archive)

print("\nAlso on disk under:")
print("  MIMIC_IV/saved_data/results/mimic_cohort_NF_30_days/time_series/")
print("    .../finetune/primenet/fig5_mimicall_all/")
print("    .../finetune/primenet/fig5_mimicall_final/")